# ComfyUI — Google Colab + Google Drive (Türkçe)

**Minik Kaşifler** için WanGP'den bağımsız ComfyUI kurulumu. Resmî kaynak: https://github.com/comfy-org/ComfyUI

- Colab'da **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
- Hücreleri **1 → 4** sırasıyla çalıştırın. Drive yetkisini yalnızca Google onay ekranında verin.
- Video modelleri **otomatik indirilmez**. ComfyUI açıldığında önce arayüzün çalıştığı doğrulanır; sonra T4'e uygun model/workflow ayrıca seçilir.
- Public bağlantı **şifreli**, geçici ve yalnızca Colab oturumu açıkken çalışır. Şifreyi veya adresi paylaşmayın.
- **WanGP dosyaları etkilenmez.** ComfyUI modelleri ve çıktıları Google Drive'da `MyDrive/ComfyUI-data` altında tutulur.
- İlk kullanımda Python bağımlılıkları yüklenir; sonraki Colab oturumlarında program tekrar kurulabilir. Google Drive'daki büyük modeller/çıktılar korunur.
- Colab'ın ücretsiz GPU sunması garanti değildir; kullanım sınırları ve oturum kapanmaları olabilir.

**Güvenlik:** Bağlantı Cloudflare Quick Tunnel üzerinden bir oturumluk HTTP Basic Auth + oturum çerezi ile korunur. Bu kişisel bir test kurulumudur; internete açık üretim sunucusu değildir.


## 1. GPU doğrulama


In [ ]:
import sys, subprocess
try:
    import torch
    print("Python:", sys.version.split()[0])
    print("PyTorch:", torch.__version__)
    print("CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("GPU algilanamadi. Calisma zamani > Calisma zamani turunu degistir > T4 GPU sec.")
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
except ImportError:
    raise RuntimeError("PyTorch bulunamadi. GPU'lu standart Google Colab calisma zamani ac.")

## 2. Google Drive bağlama ve klasörler


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
COMFY_DIR = Path('/content/ComfyUI')
DATA_DIR = Path('/content/drive/MyDrive/ComfyUI-data')
for folder in ('models/checkpoints','models/diffusion_models','models/text_encoders',
               'models/vae','models/loras','models/clip_vision','models/controlnet',
               'models/upscale_models','input','output'):
    (DATA_DIR / folder).mkdir(parents=True, exist_ok=True)
print('Drive ana klasoru:', DATA_DIR)
print('WanGP dosyalarina dokunulmaz.')

## 3. ComfyUI'yi kur ve Drive model yollarını bağla


In [ ]:
import os, sys, subprocess
from pathlib import Path
if not COMFY_DIR.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/comfy-org/ComfyUI.git',str(COMFY_DIR)],check=True)
else:
    print('ComfyUI klasoru mevcut; ayni oturumda yeniden klonlanmiyor.')
print('Gerekli ComfyUI paketleri kuruluyor...')
result = subprocess.run([sys.executable,'-m','pip','install','-r',str(COMFY_DIR/'requirements.txt'),
                         'aiohttp'],capture_output=True,text=True)
print((result.stdout+'\n'+result.stderr)[-12000:])
if result.returncode:
    raise RuntimeError('ComfyUI Python paketleri kurulamadi. Yukaridaki pip hata satirlarini gonder.')
import yaml
paths = {'comfyui': {'base_path': str(DATA_DIR),
                     'checkpoints':'models/checkpoints',
                     'diffusion_models':'models/diffusion_models',
                     'text_encoders':'models/text_encoders',
                     'vae':'models/vae','loras':'models/loras',
                     'clip_vision':'models/clip_vision',
                     'controlnet':'models/controlnet',
                     'upscale_models':'models/upscale_models'}}
(COMFY_DIR/'extra_model_paths.yaml').write_text(
    yaml.safe_dump(paths,allow_unicode=True),encoding='utf-8')
print('Kurulum tamamlandi. Drive model yollari tanimlandi.')
print('Not: Video modeli bu adimda indirilmez.')

## 4. Şifreli erişim bağlantısını oluştur ve ComfyUI'yi çalıştır

**Bu hücre çalışır durumda kalmalıdır.** Sonunda `trycloudflare.com` URL'si, kullanıcı adı ve tek oturumluk şifre yazdırılır. Bunları özel tutun. Bağlantı çalışırken Colab'ı durdurmayın.


In [ ]:
PROXY_SOURCE = "import os, secrets, hmac, asyncio, base64\nfrom aiohttp import web, ClientSession, WSMsgType\nUSER = \"comfy\"\nPASSWORD = os.environ[\"COMFY_PASSWORD\"]\nSESSION = os.environ[\"COMFY_SESSION\"]\nTARGET = \"http://127.0.0.1:8188\"\nHOP = {\"connection\",\"transfer-encoding\",\"keep-alive\",\"proxy-authenticate\",\n       \"proxy-authorization\",\"te\",\"trailers\",\"upgrade\",\"host\",\"content-length\"}\ndef authenticated(request):\n    cookie = request.cookies.get(\"comfy_session\", \"\")\n    if cookie and hmac.compare_digest(cookie,SESSION): return True\n    header = request.headers.get(\"Authorization\",\"\")\n    if header.startswith(\"Basic \"):\n        try:\n            username,password = base64.b64decode(header[6:]).decode().split(\":\",1)\n            return hmac.compare_digest(username,USER) and hmac.compare_digest(password,PASSWORD)\n        except Exception: pass\n    return False\nasync def handler(request):\n    if not authenticated(request):\n        return web.Response(status=401,text=\"ComfyUI sifresi gerekli\",\n                            headers={\"WWW-Authenticate\":'Basic realm=\"ComfyUI\"'})\n    path = request.rel_url\n    headers = {k:v for k,v in request.headers.items() if k.lower() not in HOP and k.lower()!=\"authorization\"}\n    if request.headers.get(\"Upgrade\",\"\").lower()==\"websocket\":\n        down = web.WebSocketResponse(heartbeat=30)\n        down.set_cookie(\"comfy_session\",SESSION,httponly=True,secure=True,samesite=\"Lax\")\n        await down.prepare(request)\n        async with ClientSession() as s:\n            async with s.ws_connect(TARGET+str(path),headers=headers,heartbeat=30) as up:\n                async def forward_down():\n                    async for msg in down:\n                        if msg.type==WSMsgType.TEXT: await up.send_str(msg.data)\n                        elif msg.type==WSMsgType.BINARY: await up.send_bytes(msg.data)\n                        elif msg.type in (WSMsgType.CLOSE, WSMsgType.ERROR): break\n                async def forward_up():\n                    async for msg in up:\n                        if msg.type==WSMsgType.TEXT: await down.send_str(msg.data)\n                        elif msg.type==WSMsgType.BINARY: await down.send_bytes(msg.data)\n                        elif msg.type in (WSMsgType.CLOSE, WSMsgType.ERROR): break\n                tasks=[asyncio.create_task(forward_down()),asyncio.create_task(forward_up())]\n                done,pending=await asyncio.wait(tasks,return_when=asyncio.FIRST_COMPLETED)\n                for task in pending: task.cancel()\n                await down.close()\n        return down\n    body = await request.read()\n    async with ClientSession(timeout=__import__(\"aiohttp\").ClientTimeout(total=1800)) as s:\n        async with s.request(request.method,TARGET+str(path),data=body,headers=headers,\n                             allow_redirects=False) as upstream:\n            response_headers={k:v for k,v in upstream.headers.items() if k.lower() not in HOP}\n            response = web.StreamResponse(status=upstream.status, headers=response_headers)\n            response.set_cookie(\"comfy_session\",SESSION,httponly=True,secure=True,samesite=\"Lax\")\n            await response.prepare(request)\n            async for chunk in upstream.content.iter_chunked(262144):\n                await response.write(chunk)\n            await response.write_eof()\n            return response\napp=web.Application(client_max_size=1024**3)\napp.router.add_route(\"*\",\"/{tail:.*}\",handler)\nweb.run_app(app,host=\"127.0.0.1\",port=8189,access_log=None)"

import os, sys, time, re, secrets, subprocess, urllib.request
from pathlib import Path

# Yeni oturum icin tek kullanimlik erisim sifresi.
password=secrets.token_urlsafe(16)
session=secrets.token_urlsafe(32)
proxy_file=Path('/content/comfyui_auth_proxy.py')
proxy_file.write_text(PROXY_SOURCE,encoding='utf-8')
auth_env=os.environ.copy()
auth_env.update({'COMFY_PASSWORD':password,'COMFY_SESSION':session})

# Torch/HF cache Colab yerel diskinde; modeller ve ciktilar Drive'da.
env=os.environ.copy()
env['HF_HOME']='/content/comfyui-cache/huggingface'
env['XDG_CACHE_HOME']='/content/comfyui-cache/xdg'
cmd=[sys.executable,'-u','main.py','--listen','127.0.0.1','--port','8188',
     '--lowvram','--disable-auto-launch',
     '--output-directory',str(DATA_DIR/'output'),
     '--input-directory',str(DATA_DIR/'input')]
comfy_log=open('/content/comfyui.log','w',buffering=1)
comfy=subprocess.Popen(cmd,cwd=str(COMFY_DIR),env=env,stdout=comfy_log,stderr=subprocess.STDOUT)
print('ComfyUI baslatiliyor (PID:',comfy.pid,')',flush=True)
import urllib.error
ready=False
for i in range(90):
    if comfy.poll() is not None:
        raise RuntimeError('ComfyUI erken kapandi:\n'+Path('/content/comfyui.log').read_text()[-5000:])
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats',timeout=2) as resp:
            if resp.status==200:
                ready=True
                break
    except Exception: pass
    time.sleep(2)
if not ready: raise RuntimeError('ComfyUI 180 saniyede acilmadi. /content/comfyui.log dosyasini kontrol et.')
print('ComfyUI yerel sunucu hazir.',flush=True)

proxy_log=open('/content/comfyui_proxy.log','w',buffering=1)
proxy=subprocess.Popen([sys.executable,'-u',str(proxy_file)],env=auth_env,
                       stdout=proxy_log,stderr=subprocess.STDOUT)
for _ in range(25):
    try:
        req=urllib.request.Request('http://127.0.0.1:8189/')
        urllib.request.urlopen(req,timeout=2)
    except urllib.error.HTTPError as e:
        if e.code==401: break
    except Exception: pass
    time.sleep(1)
else:
    raise RuntimeError('Sifreli proxy acilmadi. /content/comfyui_proxy.log kontrol edilmeli.')
print('Sifreli erisim kontrolu hazir.',flush=True)

binfile=Path('/content/cloudflared')
if not binfile.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',binfile)
    binfile.chmod(0o755)
tunnel_log=Path('/content/comfyui_tunnel.log')
tf=open(tunnel_log,'w',buffering=1)
tunnel=subprocess.Popen([str(binfile),'tunnel','--no-autoupdate','--url',
                         'http://127.0.0.1:8189'],stdout=tf,stderr=subprocess.STDOUT)
public_url=None
for i in range(90):
    if tunnel.poll() is not None:
        raise RuntimeError('Cloudflare tuneli kapandi:\n'+tunnel_log.read_text()[-4000:])
    log=tunnel_log.read_text(errors='replace')
    match=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',log)
    if match:
        public_url=match.group(0)
        break
    time.sleep(2)
if not public_url:
    raise RuntimeError('Cloudflare erisim adresi olusmadi:\n'+tunnel_log.read_text()[-4000:])
print('\n=================== COMFYUI ERISIM ===================')
print('Adres:',public_url)
print('Kullanici adi: comfy')
print('Sifre:',password)
print('Bu sifreyi ve baglantiyi kimseyle paylasma.')
print('====================================================\n',flush=True)
print('Hucresinin calismaya devam etmesi normaldir; sunucuyu acik tutar.')
try:
    while True:
        if comfy.poll() is not None:
            raise RuntimeError('ComfyUI kapandi:\n'+Path('/content/comfyui.log').read_text(errors='replace')[-4000:])
        if tunnel.poll() is not None:
            raise RuntimeError('Tunnel kapandi:\n'+tunnel_log.read_text(errors='replace')[-4000:])
        time.sleep(15)
except KeyboardInterrupt:
    print('ComfyUI sonlandiriliyor.')
finally:
    for p in (tunnel,proxy,comfy):
        if p.poll() is None: p.terminate()